In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft


from time import perf_counter as perf
import pyodbc
import urllib
import sqlalchemy as sqla

# pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.max_columns', None)

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')
path_congestion = os.path.join(path_main, 'Safe Equitable Resilient Infrastructure', 'Congestion')


# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'RTIS')
    path_config  = os.path.join(path_code, 'config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'RTIS')
    path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
def get_odbc_driver():
    # gets name of ODBC driver, with name "ODBC Driver <version> for SQL Server"
    drivers = [d for d in pyodbc.drivers() if 'ODBC Driver ' in d]
    
    if len(drivers) == 0:
        errmsg = f"ERROR. No usable ODBC Driver found for SQL Server." \
        f"drivers found include {drivers}. Check ODBC Administrator program" \
        "for more information."
        
        raise Exception (errmsg)
    else:
        d_versions = [re.findall('\d+', dv)[0] for dv in drivers] # [re.findall('\d+', dv)[0] for dv in drivers]
        latest_version = max([int(v) for v in d_versions])
        driver = f"ODBC Driver {latest_version} for SQL Server"
    
        return driver

def sqlqry_to_df(query_str, dbname, servername='SQL-SVR', trustedconn='yes'):   

    driver = get_odbc_driver()  

    conn_str = f"DRIVER={driver};" \
        f"SERVER={servername};" \
        f"DATABASE={dbname};" \
        f"Trusted_Connection={trustedconn}"
        
    conn_str = urllib.parse.quote_plus(conn_str)
    engine = sqla.create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str}")
       
    start_time = perf()

    # create SQL table from the dataframe
    print("Executing query. Results loading into dataframe...")
    df = pd.read_sql_query(sql=query_str, con=engine)
    rowcnt = df.shape[0]
    
    et_mins = round((perf() - start_time) / 60, 2)
    print(f"Successfully executed query in {et_mins} minutes. {rowcnt} rows loaded into dataframe.")
    
    return df

In [ ]:
db = 'NPMRDS'
qry = 'SELECT TOP 10 * FROM npmrds_2018_alltmc_trucks'

tdf = sqlqry_to_df(qry, db)
tdf

In [ ]:
print('creating query string from .sql file')
path_sql = os.path.join(path_git, 'Python Code', 'RTIS')
with open(os.path.join(path_sql, 'HoursCongestedConditions_60thPctlFFS.sql'), 'r') as query:
    query_string = query.read()

query_string

In [ ]:
db = 'NPMRDS'
qry = "SELECT * \nFROM npmrds_2018_all_tmcs_txt"

tdf = sqlqry_to_df(qry, db)
tdf

In [ ]:
db = 'NPMRDS'
qry = "SELECT TOP 5 * \nFROM npmrds_2018_alltmc_paxveh"

tdf = sqlqry_to_df(qry, db)
tdf

In [ ]:
df = sqlqry_to_df(query_string, db)

df.head()